# Часть 1 Бустинг (5 баллов)

В этой части будем предсказывать зарплату data scientist-ов в зависимости  от ряда факторов с помощью градиентного бустинга.

В датасете есть следующие признаки:



* work_year: The number of years of work experience in the field of data science.

* experience_level: The level of experience, such as Junior, Senior, or Lead.

* employment_type: The type of employment, such as Full-time or Contract.

* job_title: The specific job title or role, such as Data Analyst or Data Scientist.

* salary: The salary amount for the given job.

* salary_currency: The currency in which the salary is denoted.

* salary_in_usd: The equivalent salary amount converted to US dollars (USD) for comparison purposes.

* employee_residence: The country or region where the employee resides.

* remote_ratio: The percentage of remote work offered in the job.

* company_location: The location of the company or organization.

* company_size: The company's size is categorized as Small, Medium, or Large.

In [1]:
import pandas as pd
ds_salaries = "https://github.com/hse-ds/iad-intro-ds/raw/master/2025/homeworks/hw08_boosting_clustering/ds_salaries.csv"
df = pd.read_csv(ds_salaries)
df.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2023,SE,FT,Principal Data Scientist,80000,EUR,85847,ES,100,ES,L
1,2023,MI,CT,ML Engineer,30000,USD,30000,US,100,US,S
2,2023,MI,CT,ML Engineer,25500,USD,25500,US,100,US,S
3,2023,SE,FT,Data Scientist,175000,USD,175000,CA,100,CA,M
4,2023,SE,FT,Data Scientist,120000,USD,120000,CA,100,CA,M


## Задание 1 (0.5 балла) Подготовка



*   Разделите выборку на train, val, test (80%, 10%, 10%)
*   Выдерите salary_in_usd в качестве таргета
*   Найдите и удалите признак, из-за которого возможен лик в данных


Лик данных может произойти из-за переменной salary, так как ее значения очень похожи на значения целевой переменной.


In [2]:
X = df.drop(columns=['salary_in_usd', 'salary'])
y = df['salary_in_usd']

In [3]:
from sklearn.model_selection import train_test_split

x_train, x_t, y_train, y_t = train_test_split(X, y, test_size=0.2, random_state=42)

x_test, x_val, y_test, y_val = train_test_split(x_t, y_t, test_size=0.5, random_state=42)

## Задание 2 (0.5 балла) Линейная модель


*   Закодируйте категориальные  признаки с помощью OneHotEncoder
*   Обучите модель линейной регрессии
*   Оцените  качество через MAPE и RMSE


In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder

categorical_features = x_train.select_dtypes(include=['object', 'category']).columns.tolist()
one_hot_encoder = OneHotEncoder(handle_unknown='ignore', drop="first")

x_train_cat = pd.DataFrame(
    data = one_hot_encoder.fit_transform(x_train[categorical_features]).toarray(),
    columns = one_hot_encoder.get_feature_names_out(categorical_features))
x_train_num = x_train.select_dtypes(exclude=['object'])
x_train = pd.concat([x_train_cat.set_index(x_train.index), x_train_num], axis=1)

x_test_cat = pd.DataFrame(
    data = one_hot_encoder.transform(x_test[categorical_features]).toarray(),
    columns = one_hot_encoder.get_feature_names_out(categorical_features))
x_test_num = x_test.select_dtypes(exclude=['object'])
x_test = pd.concat([x_test_cat.set_index(x_test.index), x_test_num], axis=1)

x_val_cat = pd.DataFrame(
    data = one_hot_encoder.transform(x_val[categorical_features]).toarray(),
    columns = one_hot_encoder.get_feature_names_out(categorical_features))
x_val_num = x_val.select_dtypes(exclude=['object'])
x_val = pd.concat([x_val_cat.set_index(x_val.index), x_val_num], axis=1)

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2, 3, 4, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2, 4, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [5]:
lin_reg = LinearRegression()
lin_reg.fit(x_train, y_train)
y_pred = lin_reg.predict(x_test)
mape = mean_absolute_percentage_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** (0.5)

print('MAPE: ', mape)
print('RMSE: ', rmse)

MAPE:  0.43164047671758754
RMSE:  46984.19170454998


Значение MAPE, равное 0.43, говорит о достаточно сильном отклонении прогнозов модели от фактических данных. Это подтверждает RMSE = 46984 долларов.

## Задание 3 (0.5 балла) XGboost

Начнем с библиотеки xgboost.

Обучите модель `XGBRegressor` на тех же данных, что линейную модель, подобрав оптимальные гиперпараметры (`max_depth, learning_rate, n_estimators, gamma`, etc.) по валидационной выборке. Оцените качество итоговой модели (MAPE, RMSE), скорость обучения и скорость предсказания.

In [6]:
predict_time = []
train_time = []
mapes = []
rmses = []

In [7]:
from xgboost.sklearn import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, mean_absolute_percentage_error, mean_squared_error
import time
import numpy as np

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 200, 300],
    'gamma': [0, 0.01, 0.1]
}

model_xgb = XGBRegressor(random_state=42)
grid_search = GridSearchCV(model_xgb, param_grid, cv=3,
                           scoring='neg_mean_absolute_error')

grid_search.fit(x_train, y_train)
best_params = grid_search.best_params_

print(f"Лучшие гиперпараметры: {best_params}")

Лучшие гиперпараметры: {'gamma': 0, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 100}


In [8]:
best_gamma = best_params['gamma']
best_learning_rate =  best_params['learning_rate']
best_max_depth =  best_params['max_depth']
best_n_estimators =  best_params['n_estimators']

In [9]:
model_xgb = XGBRegressor(gamma = best_gamma, learning_rate = best_learning_rate, max_depth = best_max_depth, n_estimators = best_n_estimators, random_state=42)
start_t = time.time()
model_xgb.fit(x_train, y_train)
end_t = time.time()
time_for_train = end_t - start_t
print(f"Время обучения: {time_for_train:.2f} секунд")
train_time.append(time_for_train)

Время обучения: 0.35 секунд


In [10]:
start_t = time.time()
y_pred = model_xgb.predict(x_test)
end_t = time.time()
time_for_predict = end_t - start_t

mape = mean_absolute_percentage_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Время предсказания: {time_for_predict:.4f} секунд")
print('MAPE: ', mape)
print('RMSE: ', rmse)
predict_time.append(time_for_predict)
rmses.append(rmse)
mapes.append(mape)

Время предсказания: 0.0557 секунд
MAPE:  0.3960414230823517
RMSE:  46211.05668560285


Ошибка MAPE в результате применения XGBRegressor стала меньше на 0.04, а RMSE снизилось на 46984 - 46211 = 773 доллара, поэтому можно сказать, что данная модель лучше линейной, однако ошибки все еще большие.

## Задание 4 (1 балл) CatBoost

Теперь библиотека CatBoost.

Обучите модель `CatBoostRegressor`, подобрав оптимальные гиперпараметры (`depth, learning_rate, iterations`, etc.) по валидационной выборке. Оцените качество итоговой модели (MAPE, RMSE), скорость обучения и скорость предсказания.

In [11]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 10.2 MB/s eta 0:00:00


In [12]:
from catboost import CatBoostRegressor

param_grid = {
    'depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'iterations': [100, 200, 300],
}

model_cbr = CatBoostRegressor(random_seed=42, verbose=0)
grid_search = GridSearchCV(model_cbr, param_grid, cv=3, scoring='neg_mean_absolute_error')

grid_search.fit(x_train, y_train, verbose=0)

best_params = grid_search.best_params_
print(f"Лучшие гиперпараметры: {best_params}")

Лучшие гиперпараметры: {'depth': 5, 'iterations': 300, 'learning_rate': 0.1}


In [13]:
best_learning_rate =  best_params['learning_rate']
best_depth =  best_params['depth']
best_iterations =  best_params['iterations']

In [14]:
model_cbr = CatBoostRegressor(learning_rate = best_learning_rate, depth = best_depth, iterations = best_iterations, random_state=42, verbose=0)
start_t = time.time()
model_cbr.fit(x_train, y_train, verbose=0)
end_t = time.time()
time_for_train = end_t - start_t
print(f"Время обучения: {time_for_train:.2f} секунд")
train_time.append(time_for_train)

Время обучения: 0.47 секунд


In [15]:
start_t = time.time()
pred = model_cbr.predict(x_test)
end_t = time.time()
time_for_predict = end_t - start_t

test_mape = mean_absolute_percentage_error(y_test, pred)
test_rmse = np.sqrt(mean_squared_error(y_test, pred))

print(f"Время предсказания: {time_for_predict:.4f} секунд")
print('MAPE: ', mape)
print('RMSE: ', rmse)
predict_time.append(time_for_predict)
rmses.append(rmse)
mapes.append(mape)

Время предсказания: 0.0059 секунд
MAPE:  0.3960414230823517
RMSE:  46211.05668560285


Применение CatBoost не лало улучшений с точки зрезния ошибок MAPE и RMSE по сравнению с XGBRegressor. Времени на обучение для CatBoost затрачено больше(0.71), чем для XGBRegressor(0.25), но предсказание работает быстрее - 0.0043(у XGBRegressor это заняло  0.0780 секунд)

Для применения catboost моделей не обязательно сначала кодировать категориальные признаки, модель может кодировать их сама. Обучите catboost с подбором оптимальных гиперпараметров снова, используя pool для передачи данных в модель с указанием какие признаки категориальные, а какие нет с помощью параметра cat_features. Оцените качество и время. Стало ли лучше?

In [16]:
ds_salaries = "https://github.com/hse-ds/iad-intro-ds/raw/master/2025/homeworks/hw08_boosting_clustering/ds_salaries.csv"
df = pd.read_csv(ds_salaries)
df.head()
X = df.drop(columns=['salary_in_usd', 'salary'])
y = df['salary_in_usd']
x_train, x_t, y_train, y_t = train_test_split(X, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_t, y_t, test_size=0.5, random_state=42)

In [17]:
from catboost import Pool
cat_features_indices = np.where(x_train.dtypes == 'object')[0].tolist()

pool_train = Pool(data=x_train, label=y_train, cat_features=cat_features_indices)
pool_test = Pool(data=x_test, cat_features=cat_features_indices)

model_cbr = CatBoostRegressor(learning_rate = best_learning_rate, depth = best_depth, iterations = best_iterations, random_state=42, verbose=0)
start_t = time.time()
model_cbr.fit(pool_train)
end_t = time.time()
time_for_train = end_t - start_t
print(f"Время обучения: {time_for_train:.2f} секунд")
train_time.append(time_for_train)

Время обучения: 3.44 секунд


In [18]:
start_t = time.time()
preds = model_cbr.predict(pool_test)
end_t = time.time()
time_for_predict = end_t - start_t
print("Время предсказания:", time_for_predict, "секунды")

mape = mean_absolute_percentage_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print("MAPE:", mape)
print("RMSE:", rmse)
predict_time.append(time_for_predict)
rmses.append(rmse)
mapes.append(mape)

Время предсказания: 0.0020117759704589844 секунды
MAPE: 0.42613963261062354
RMSE: 45169.023683833846


**Ответ:** скорость предсказания стала быстрее по сравнению с предыдущей моделью, но метрики ухудшились.

## Задание 5 (0.5 балла) LightGBM

И наконец библиотека LightGBM - используйте `LGBMRegressor`, снова подберите гиперпараметры, оцените качество и скорость.


In [19]:
ds_salaries = "https://github.com/hse-ds/iad-intro-ds/raw/master/2025/homeworks/hw08_boosting_clustering/ds_salaries.csv"
df = pd.read_csv(ds_salaries)
df.head()
X = df.drop(columns=['salary_in_usd', 'salary'])
y = df['salary_in_usd']
x_train, x_t, y_train, y_t = train_test_split(X, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_t, y_t, test_size=0.5, random_state=42)

In [20]:
categorical_features = x_train.select_dtypes(include=['object', 'category']).columns.tolist()
one_hot_encoder = OneHotEncoder(handle_unknown='ignore', drop="first")

x_train_cat = pd.DataFrame(
    data = one_hot_encoder.fit_transform(x_train[categorical_features]).toarray(),
    columns = one_hot_encoder.get_feature_names_out(categorical_features))
x_train_num = x_train.select_dtypes(exclude=['object'])
x_train = pd.concat([x_train_cat.set_index(x_train.index), x_train_num], axis=1)

x_test_cat = pd.DataFrame(
    data = one_hot_encoder.transform(x_test[categorical_features]).toarray(),
    columns = one_hot_encoder.get_feature_names_out(categorical_features))
x_test_num = x_test.select_dtypes(exclude=['object'])
x_test = pd.concat([x_test_cat.set_index(x_test.index), x_test_num], axis=1)

x_val_cat = pd.DataFrame(
    data = one_hot_encoder.transform(x_val[categorical_features]).toarray(),
    columns = one_hot_encoder.get_feature_names_out(categorical_features))
x_val_num = x_val.select_dtypes(exclude=['object'])
x_val = pd.concat([x_val_cat.set_index(x_val.index), x_val_num], axis=1)

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2, 3, 4, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2, 4, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [21]:
from lightgbm import LGBMRegressor


param_grid = {
    'max_depth' : [3, 5, 7, 10, 15],
    'learning_rate' : [0.1, 0.2, 0.5],
    'n_estimators' : [100, 200, 300],
    'min_data_in_leaf': [20, 30, 50],
    'verbosity': [-1]
}

model_lgbm = LGBMRegressor()

grid_search = GridSearchCV(model_lgbm, param_grid, cv=3, scoring='neg_mean_absolute_error', verbose = 0)

grid_search.fit(x_train, y_train)

best_params = grid_search.best_params_

print(f"Лучшие гиперпараметры: {best_params}")

Лучшие гиперпараметры: {'learning_rate': 0.2, 'max_depth': 3, 'min_data_in_leaf': 20, 'n_estimators': 100, 'verbosity': -1}


In [22]:
best_learning_rate =  best_params['learning_rate']
best_max_depth =  best_params['max_depth']
best_n_estimators =  best_params['n_estimators']
best_data_in_leaf =  best_params['min_data_in_leaf']

In [23]:
model_lgbm = LGBMRegressor(learning_rate = best_learning_rate, max_depth = best_max_depth, n_estimators = best_n_estimators, min_data_in_leaf = best_data_in_leaf, random_state=42, verbosity = [-1], verbose=0)
start_t = time.time()
model_lgbm.fit(x_train, y_train)
end_t = time.time()
time_for_train = end_t - start_t
print(f"Время обучения: {time_for_train:.2f} секунд")
train_time.append(time_for_train)

start_t = time.time()
y_pred = model_lgbm.predict(x_test)
end_t = time.time()
time_for_predict = end_t - start_t

mape = mean_absolute_percentage_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Время предсказания:", time_for_predict, "секунды")
print('MAPE: ', mape)
print('RMSE: ', rmse)
predict_time.append(time_for_predict)
rmses.append(rmse)
mapes.append(mape)

Время обучения: 0.07 секунд
Время предсказания: 0.007721662521362305 секунды
MAPE:  0.4067409670318092
RMSE:  46464.39454252943


## Задание 6 (2 балла) Сравнение и выводы

Сравните модели бустинга и сделайте про них выводы, какая из моделей показала лучший/худший результат по качеству, скорости обучения и скорости предсказания? Как отличаются гиперпараметры для разных моделей?

In [24]:
data = {
    'Модель': ['XGBReagressor', 'CatBoost', 'CatBoost with categories', 'LGBMRegressor'],
    'Время обучения': train_time,
    'Время предсказания': predict_time,
    'MAPE': mapes,
    'RMSE': rmses
}

df = pd.DataFrame(data)
display(df)

,Модель,Время обучения,Время предсказания,MAPE,RMSE
0,XGBReagressor,0.351559,0.055669,0.396041,46211.056686
1,CatBoost,0.470267,0.005881,0.396041,46211.056686
2,CatBoost with categories,3.435556,0.002012,0.426140,45169.023684
3,LGBMRegressor,0.068359,0.007722,0.406741,46464.394543


**Ответ:**
Лучшими моделями по качеству стали XGBReagressor и CatBoost. Они имеют одинаковые ошибки, которые являются наименьшими среди всех моделей. Худшая модель - LGBMRegressor.

Лучшей моделью с точки зрения скорости обучения стала LGBMRegressor. Худшая модель - CatBoost with categories.

Лучшей моделью с точки зрения скорости предсказания стала CatBoost with categories. Худшая - XGBReagressor.

XGBReagressor имеет оптимальные параметры:
gamma = 0,
learning_rate = 0.1,
max_depth = 7,
n_estimators = 100

CatBoost имеет оптимальные параметры:
depth = 5,
iterations = 300,
learning_rate = 0.1

LGBMRegressor имеет оптимальные параметры:
learning_rate = 0.2,
max_depth = 3,
min_data_in_leaf = 20,
n_estimators = 100

# Часть 2 Кластеризация (5 баллов)

Будем работать с данными о том, каких исполнителей слушают пользователи музыкального сервиса.

Каждая строка таблицы - информация об одном пользователе. Каждый столбец - это исполнитель (The Beatles, Radiohead, etc.)

Для каждой пары (пользователь, исполнитель) в таблице стоит число - доля прослушивания этого исполнителя этим пользователем.


In [25]:
import pandas as pd
ratings = pd.read_excel("https://github.com/evgpat/edu_stepik_rec_sys/blob/main/datasets/sample_matrix.xlsx?raw=true", engine='openpyxl')
ratings.head()

,user,the beatles,radiohead,deathcab for cutie,coldplay,modest mouse,sufjan stevens,dylan. bob,red hot clili peppers,pink fluid,...,municipal waste,townes van zandt,curtis mayfield,jewel,lamb,michal w. smith,群星,agalloch,meshuggah,yellowcard
0,0,NaN,0.020417,NaN,NaN,NaN,NaN,NaN,0.030496,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,NaN,0.184962,0.024561,NaN,NaN,0.136341,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,NaN,NaN,0.028635,NaN,NaN,NaN,0.024559,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,0.043529,0.086281,0.034590,0.016712,0.015935,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Будем строить кластеризацию исполнителей: если двух исполнителей слушало много людей примерно одинаковую долю своего времени (то есть векторы близки в пространстве), то, возможно исполнители похожи. Эта информация может быть полезна при построении рекомендательных систем.

## Задание 1 (0.5 балла) Подготовка

Транспонируем матрицу ratings, чтобы по строкам стояли исполнители.

In [47]:
T_ratings = ratings.T
T_ratings.head()

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
user,0.000000,1.000000,2.000000,3.0,4.000000,5.000000,6.0,7.0,8.000000,9.000000,...,4990.000000,4991.0,4992.000000,4993.000000,4994.000000,4995.000000,4996.0,4997.000000,4998.0,4999.000000
the beatles,NaN,NaN,NaN,NaN,0.043529,NaN,NaN,NaN,0.093398,0.017621,...,NaN,NaN,0.121169,0.038168,0.007939,0.017884,NaN,0.076923,NaN,NaN
radiohead,0.020417,0.184962,NaN,NaN,0.086281,0.006322,NaN,NaN,NaN,0.019156,...,0.017735,NaN,NaN,NaN,0.011187,NaN,NaN,NaN,NaN,NaN
deathcab for cutie,NaN,0.024561,0.028635,NaN,0.034590,NaN,NaN,NaN,NaN,0.013349,...,0.121344,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.027893
coldplay,NaN,NaN,NaN,NaN,0.016712,NaN,NaN,NaN,NaN,NaN,...,0.217175,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Выкиньте строку под названием `user`.

In [48]:
T_ratings.drop(index=["user"], inplace=True)
T_ratings.head()

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
the beatles,NaN,NaN,NaN,NaN,0.043529,NaN,NaN,NaN,0.093398,0.017621,...,NaN,NaN,0.121169,0.038168,0.007939,0.017884,NaN,0.076923,NaN,NaN
radiohead,0.020417,0.184962,NaN,NaN,0.086281,0.006322,NaN,NaN,NaN,0.019156,...,0.017735,NaN,NaN,NaN,0.011187,NaN,NaN,NaN,NaN,NaN
deathcab for cutie,NaN,0.024561,0.028635,NaN,0.034590,NaN,NaN,NaN,NaN,0.013349,...,0.121344,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.027893
coldplay,NaN,NaN,NaN,NaN,0.016712,NaN,NaN,NaN,NaN,NaN,...,0.217175,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
modest mouse,NaN,NaN,NaN,NaN,0.015935,NaN,NaN,NaN,NaN,0.030437,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


В таблице много пропусков, так как пользователи слушают не всех-всех исполнителей, чья музыка представлена в сервисе, а некоторое подмножество (обычно около 30 исполнителей)


Доля исполнителя в музыке, прослушанной  пользователем, равна 0, если пользователь никогда не слушал музыку данного музыканта, поэтому заполните пропуски нулями.



In [49]:
T_ratings.fillna(0, inplace=True)
T_ratings.reset_index(inplace=True)
T_ratings.rename(columns={'index': 'group'}, inplace=True)
T_ratings.sample()

,group,0,1,2,3,4,5,6,7,8,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
931,日dir en grey,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Задание 2 (0.5 балла) Первая кластеризация

Примените KMeans с 5ю кластерами, сохраните полученные лейблы

In [50]:
from sklearn.cluster import KMeans

model_km = KMeans(n_clusters = 5)
model_km.fit(T_ratings.iloc[:, 1:])

KMeans(n_clusters=5)

Выведите размеры кластеров. Полезной ли получилась кластеризация? Почему KMeans может выдать такой результат?

In [51]:
from collections import Counter
labels = model_km.labels_
ratings_with_clusters = T_ratings
ratings_with_clusters['cluster_label'] = model_km.labels_
size = Counter([label.item() for label in labels])
print(dict(size))

{3: 1, 1: 996, 4: 1, 2: 1, 0: 1}


**Ответ:** кластеризация получилась бесполезной, поскольку появляются кластеры, включающие только один объект. Возможно, в выборке содержатся выбросы и KMeans определяет их в отдельные кластеры.

## Задание 3 (0.5 балла) Объяснение результатов

При кластеризации получилось $\geq 1$ кластера размера 1. Выведите исполнителей, которые составляют такие кластеры. Среди них должна быть группа The Beatles.

In [52]:
clusters_with_one_object = [key for key, value in size.items() if value == 1]
clusters_with_one_object

[3, 4, 2, 0]

In [53]:
artists =  ratings_with_clusters[ratings_with_clusters['cluster_label'].isin(clusters_with_one_object)]
artists

,group,0,1,2,3,4,5,6,7,8,...,4991,4992,4993,4994,4995,4996,4997,4998,4999,cluster_label
0,the beatles,0.0,0.0,0.0,0.0,0.043529,0.0,0.0,0.0,0.093398,...,0.0,0.121169,0.038168,0.007939,0.017884,0.0,0.076923,0.0,0.0,3
8,pink fluid,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.020780,0.000000,0.000000,0.0,0.000000,0.0,0.0,4
70,boards of canada,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.000000,0.033923,0.000000,0.0,0.000000,0.0,0.0,2
658,jonas brothers,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0


Изучите данные, почему именно The Beatles выделяется?

Подсказка: посмотрите на долю пользователей, которые слушают каждого исполнителя, среднюю долю прослушивания.

In [61]:
number_of_all_users = len(ratings)
one_cluster_artists = artists.drop(columns=['group', 'cluster_label'])
number_of_listeners = one_cluster_artists.astype(bool).sum(axis=1)
p = number_of_listeners / number_of_all_users
result = pd.concat([artists['group'], p.rename('proportion of listeners')], axis=1)
result

,group,proportion of listeners
0,the beatles,0.3342
8,pink fluid,0.1256
70,boards of canada,0.0642
658,jonas brothers,0.0138


In [55]:
mean_proportion = result['proportion of listeners'].mean()
print(f'Средняя доля прослушиваний песен, которые попали в отдельный кластер: {mean_proportion}')

Средняя доля прослушиваний песен, которые попали в отдельный кластер: 0.1348


In [62]:
number_of_all_users = len(ratings)
all_artists = ratings_with_clusters.drop(columns=['group', 'cluster_label'])
number_of_listeners = all_artists.astype(bool).sum(axis=1)
p = number_of_listeners / number_of_all_users
result = pd.concat([ratings_with_clusters['group'], p.rename('proportion of listeners')], axis=1)
result.head()

,group,proportion of listeners
0,the beatles,0.3342
1,radiohead,0.2778
2,deathcab for cutie,0.1862
3,coldplay,0.1682
4,modest mouse,0.1628


In [63]:
mean_proportion = result['proportion of listeners'].mean()
print(f'Средняя доля прослушиваний музыкальных групп: {mean_proportion}')

Средняя доля прослушиваний музыкальных групп: 0.026826200000000005


**Ответ:** группа The Beatles может выделяться в отдельный кластер из-за высокой доли пользователей, слушающих эту группу. Поскольку группа достаточно популярна у нее может быть больше прослушиваний, нежели у других муз. исполнителей. Доля прослушиваний у битлз равна 0.3342, что очень сильно превышает среднюю долю прослушиваний среди всех исполнителей в датасете - 0.0268. The Beatles можно назвать выбросом и они выделяются в отдельный кластер.

## Задание 4 (0.5 балла) Улучшение кластеризации

Попытаемся избавиться от этой проблемы: нормализуйте данные при помощи `normalize`.

In [56]:
from sklearn.preprocessing import normalize

groups_names = T_ratings['group']
number_columns = T_ratings.drop(columns=['group', 'cluster_label'])
normalize_ratings = normalize(number_columns)
df = pd.DataFrame(normalize_ratings, columns=number_columns.columns)
df.insert(0, 'group', groups_names)

In [57]:
df.head()

,group,0,1,2,3,4,5,6,7,8,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
0,the beatles,0.000000,0.000000,0.000000,0.0,0.012054,0.000000,0.0,0.0,0.025864,...,0.000000,0.0,0.033554,0.010569,0.002199,0.004952,0.0,0.021302,0.0,0.000000
1,radiohead,0.009348,0.084688,0.000000,0.0,0.039505,0.002894,0.0,0.0,0.000000,...,0.008120,0.0,0.000000,0.000000,0.005122,0.000000,0.0,0.000000,0.0,0.000000
2,deathcab for cutie,0.000000,0.017278,0.020144,0.0,0.024333,0.000000,0.0,0.0,0.000000,...,0.085361,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.019622
3,coldplay,0.000000,0.000000,0.000000,0.0,0.011129,0.000000,0.0,0.0,0.000000,...,0.144628,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000
4,modest mouse,0.000000,0.000000,0.000000,0.0,0.010260,0.000000,0.0,0.0,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000


Примените KMeans с 5ю кластерами на преобразованной матрице, посмотрите на их размеры. Стало ли лучше? Может ли кластеризация быть полезной теперь?

In [58]:
model_km = KMeans(n_clusters = 5)
model_km.fit(df.iloc[:, 1:])

KMeans(n_clusters=5)

In [59]:
labels = model_km.labels_
ratings_with_clusters = df
ratings_with_clusters['cluster_label'] = model_km.labels_
size = Counter([label.item() for label in labels])
print(dict(size))

{4: 413, 1: 161, 3: 173, 2: 152, 0: 101}


**Ответ:** после нормализации данных модель стала лучше разделять исполнителей на кластеры. Масштабирование позволило снизить влияние выбросов на разделение. Кластеры стали сбалансированными (кроме 4 кластера, в котором в 3 раза больше объектов нежели в других).

## Задание 5 (1 балл) Центроиды

Выведите для каждого кластера названия топ-10 исполнителей, ближайших к центроиду по косинусной мере. Проинтерпретируйте результат. Что можно сказать о смысле кластеров?

In [66]:
from scipy.spatial.distance import cosine
centroids = model_km.cluster_centers_

ratings_with_clusters = df.select_dtypes(include=np.number).copy()
ratings_with_clusters['cluster_label'] = labels
groups_names = df['group']
artists_near_cluster = {}

for label in range(len(centroids)):
    current = ratings_with_clusters[labels == label].drop(columns=['cluster_label']).values
    dist = []
    centroid = centroids[label]

    for p in current:
        dist.append(cosine(p, centroid))

    top_ten_indices = np.argsort(dist)[:10]
    group_names = groups_names[labels == label]
    artists_near_cluster[label] = group_names.iloc[top_ten_indices].values.tolist()

for label, artists in artists_near_cluster.items():
    print(f"Топ 10 артистов в кластере с номером {label}: {', '.join(map(str, artists))}")


Топ 10 артистов в кластере с номером 0: metallica, as i lay dying, iron maiden, killswitch engage, megadeth, tool, in flames, slayer, koЯn, system of a down
Топ 10 артистов в кластере с номером 1: fall out boy, paramore, the all-americian rejects, dashboard confesssional, cartel, anberlin, kelly clarkson, the fray, the red jumpsuit apparatus, mayday parade
Топ 10 артистов в кластере с номером 2: nas, jay-z, kanye west, a tribe called quest, the roots featuring d'angelo, lupe the gorilla, mos def, gangstarr, de la soul, little brother
Топ 10 артистов в кластере с номером 3: less than jake, rancid, greenday, against me!, streetlight manifesto, acdc, descendents, led zeppelin., blink-182, the ramones
Топ 10 артистов в кластере с номером 4: radiohead, the arcade fire, sufjan stevens, belle and sebastian, the shins, broken social scene, animal collective, the beatles, the pixies, deathcab for cutie


**Ответ:**

Рассмотрим каждый кластер подробнее:

1. Кластер с номером 0.

metallica - жанр: трэш-метал, спид-метал, хард-рок

as i lay dying - жанр: 	мелодичный-металкор, мелодичный дэт-метал

killswitch engage - жанр: металкор, мелодичный металкор

(данные про жанры взяты из интернета)

Видно, что в нулевом кластере собраны артисты, основным жанром музыки для которых является металл.

2. Кластер с номером 1.

dashboard confesssional - жанр: эмо, инди-рок

the fray - Жанр: Христианский рок, поп-рок, Альтернативный рок

the all-americian rejects - альтернативный рок, поп-рок, пауэр-поп, поп-панк, панк-рок, эмо

Аналогично, жанры схожи.

3. Кластер с номером 2.

jay-z - жанр: рэп, хип-хоп

kanye west - жанр: хип-хоп, рэп, поп

de la soul - жанр: хип-хоп, рэп

4. Кластер с номером 3.

less than jake - жанр: ска - панк, поп-панк, панк - рок, скейтборд-панк

against me! - жанр: панк-рок, фол-панк

и тд.

Таким образом, можно заметить, что каждый кластер содержит наиболее схожих музыкальных исполнителей, которые работают в схожих жанрах.

## Задание 6 (1 балл) Визуализация

Хотелось бы как-то визуализировать полученную кластеризацию. Постройте точечные графики `plt.scatter` для нескольких пар признаков исполнителей, покрасив точки в цвета кластеров. Почему визуализации получились такими? Хорошо ли они отражают разделение на кластеры? Почему?

In [ ]:
import matplotlib.pyplot as plt

# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

Для визуализации данных высокой размерности существует метод t-SNE (стохастическое вложение соседей с t-распределением). Данный метод является нелинейным методом снижения размерности: каждый объект высокой размерности будет моделироваться объектов более низкой (например, 2) размерности таким образом, чтобы похожие объекты моделировались близкими, непохожие - далекими с большой вероятностью.

Примените `TSNE` из библиотеки `sklearn` и визуализируйте полученные объекты, покрасив их в цвета их кластеров

In [ ]:
from sklearn.manifold import TSNE

# -- YOUR CODE HERE --

## Задание 7 (1 балл) Подбор гиперпараметров

Подберите оптимальное количество кластеров (максимум 100 кластеров) с использованием индекса Силуэта. Зафиксируйте `random_state=42`

In [ ]:
from sklearn.metrics import silhouette_score

# -- YOUR CODE HERE --

Выведите исполнителей, ближайших с центроидам (аналогично заданию 5). Как соотносятся результаты? Остался ли смысл кластеров прежним? Расскажите про смысл 1-2 интересных кластеров, если он изменился и кластеров слишком много, чтобы рассказать про все.

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

Сделайте t-SNE визуализацию полученной кластеризации.

In [ ]:
# -- YOUR CODE HERE --

Если кластеров получилось слишком много и визуально цвета плохо отличаются, покрасьте только какой-нибудь интересный кластер из задания выше (`c = (labels == i)`). Хорошо ли этот кластер отражается в визуализации?

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --